In [0]:
%pip install azure-storage-blob
dbutils.library.restartPython()

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.aemo_landing")
print("Volume ready")

In [0]:
import os
from azure.storage.blob import ContainerClient

ACCOUNT = "stnemshreyas01"
CONTAINER = "raw"
PREFIX = "aemo/price_demand/"
LANDING = "/Volumes/workspace/bronze/aemo_landing"

sas = dbutils.secrets.get("nem", "adls_raw_sas")
container = ContainerClient.from_container_url(
    f"https://{ACCOUNT}.blob.core.windows.net/{CONTAINER}?{sas}"
)

copied, skipped = 0, 0
for blob in container.list_blobs(name_starts_with=PREFIX):
    if not blob.name.lower().endswith(".csv"):
        continue  # skip folder markers
    target = os.path.join(LANDING, blob.name)
    # Idempotent: skip files we already have with the same size
    if os.path.exists(target) and os.path.getsize(target) == blob.size:
        skipped += 1
        continue
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open(target, "wb") as f:
        f.write(container.download_blob(blob.name).readall())
    copied += 1

print(f"Copied {copied} new files, skipped {skipped} already present")

In [0]:
files = [
    os.path.join(root, f)
    for root, _, names in os.walk(f"{LANDING}/aemo/price_demand")
    for f in names if f.endswith(".csv")
]
months = sorted({p.split("month=")[1][:6] for p in files})
print(len(files), "files across", len(months), "months:", months[0], "to", months[-1])